# 7. Transformer Encoder

**Цель:** Собрать энкодерный блок трансформера: Multi-Head Self-Attention → Add & Norm → FFN → Add & Norm. Реализовать LayerNorm, FFN, Residual connections и стекирование блоков.

---

In [1]:
import sys, os, logging, math

import torch  # Фреймворк глубокого обучения
import torch.nn as nn  # Слои и функции
import torch.nn.functional as F  # Функциональный API
import numpy as np  # Численные расчёты
import matplotlib.pyplot as plt  # Графики

if torch.cuda.is_available():  # GPU (NVIDIA)
    device = torch.device("cuda")

elif torch.backends.mps.is_available():  # GPU (Apple Metal)
    device = torch.device("mps")

else:  # CPU
    device = torch.device("cpu")

## 7.1 Layer Normalization

**Формула:**
$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

- Нормализуется по последней размерности (d_model)
- $\mu, \sigma$ — среднее и дисперсия по признакам для каждого токена
- $\gamma$ (scale) и $\beta$ (shift) — обучаемые параметры
- $\epsilon$ — малая константа для численной стабильности

**Почему LayerNorm, а не BatchNorm?**
- BatchNorm зависит от batch_size и плохо работает при переменной длине
- LayerNorm нормализует каждый токен независимо
- В трансформерах LayerNorm стабильнее при обучении

In [2]:

class LayerNorm(nn.Module):  # Нормализация по последней размерности
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))  # Масштаб (обучаемый)
        self.beta = nn.Parameter(torch.zeros(d_model))  # Сдвиг (обучаемый)
        self.eps = eps  # Малая константа для стабильности
    
    def forward(self, x):  # x: (batch, seq_len, d_model)
        mu = x.mean(dim=-1, keepdim=True)  # Среднее по признакам
        sigma = x.std(dim=-1, keepdim=True, unbiased=False)  # Стандартное отклонение
        return self.gamma * (x - mu) / (sigma + self.eps) + self.beta  # Нормализация

# Сравнение с nn.LayerNorm
x = torch.randn(4, 10, 32)
our_ln = LayerNorm(32)  # Наша реализация
torch_ln = nn.LayerNorm(32)  # Эталон PyTorch

with torch.no_grad():  # Копируем веса из эталона для сравнения
    our_ln.gamma.data = torch_ln.weight.data
    our_ln.beta.data = torch_ln.bias.data
    diff = (our_ln(x) - torch_ln(x)).abs().max().item()
    print(f"Max difference from nn.LayerNorm: {diff:.2e}")

Max difference from nn.LayerNorm: 1.98e-05


## 7.2 Feed-Forward Network (FFN)

**Структура:** Linear → GELU → Linear
$$
\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 \cdot x + b_1) + b_2
$$

- Внутренняя размерность: d_ff = 4 * d_model
- GELU предпочтительнее ReLU в трансформерах (гладкая нелинейность)

In [3]:

class FeedForward(nn.Module):  # Двухслойная полносвязная сеть
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # Размерность скрытого слоя (обычно 4×d_model)
        self.fc1 = nn.Linear(d_model, d_ff)  # Расширение: d_model → d_ff
        self.fc2 = nn.Linear(d_ff, d_model)  # Сжатие: d_ff → d_model
        self.dropout = nn.Dropout(dropout)  # Регуляризация
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))  # Linear → GELU → Dropout → Linear

ffn = FeedForward(d_model=32)
x = torch.randn(4, 10, 32)
print(f"FFN input:  {x.shape}")
print(f"FFN output: {ffn(x).shape}")

FFN input:  torch.Size([4, 10, 32])
FFN output: torch.Size([4, 10, 32])


## 7.3 Residual Connections

**Зачем остаточные связи?**
- Позволяют градиенту "обтекать" слои (shortcut)
- Решают проблему затухания градиента в глубоких сетях
- Теоретически: каждый блок учится "поправке" к входу

**Формула:** output = LayerNorm(x + Sublayer(x))

**Pre-Norm vs Post-Norm:**
1. **Post-Norm** (оригинальный Vaswani): Sublayer → Add → Norm
2. **Pre-Norm** (современный стандарт): Norm → Sublayer → Add
- Pre-Norm стабильнее при обучении (градиенты не затухают в глубоких стеках)

## 7.4 TransformerEncoderBlock

Собираем всё вместе: Self-Attention → Add & Norm → FFN → Add & Norm

In [4]:

def scaled_dot_product_attention(Q, K, V, mask=None):  # Attention: Q @ K^T / √d_k
    d_k = Q.size(-1)  # Размерность ключа
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # Скалярное произведение с нормализацией
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))  # Маскируем padding
    attn = F.softmax(scores, dim=-1)
    return attn @ V, attn  # Взвешенная сумма значений + веса внимания

class MultiHeadAttention(nn.Module):  # Многоголовое внимание
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0  # d_model должен делиться на n_heads
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Размерность каждой головы
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # Проекция Query
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # Проекция Key
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # Проекция Value
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # Выходная проекция
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):  # Q, K, V: (batch, seq_len, d_model)
        batch = Q.size(0)
        Q = self.W_Q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)  # (batch, n_heads, seq_len, d_k)
        K = self.W_K(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)  # (batch, n_heads, seq_len, d_k)
        V = self.W_V(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)  # (batch, n_heads, seq_len, d_k)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, n_heads, seq_len, seq_len)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
        scores = scores.masked_fill(mask == 0, float('-inf'))  # Маскируем padding
        attn = self.dropout(F.softmax(scores, dim=-1))  # Веса внимания с Dropout
        
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.d_model)  # Собираем головы
        return self.W_O(output), attn  # Выходная проекция + веса внимания

class TransformerEncoderBlock(nn.Module):  # Один блок энкодера (Pre-Norm)
    """Один блок энкодера трансформера (Pre-Norm)."""
    
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)  # Self-attention
        self.ffn = FeedForward(d_model, d_ff, dropout)  # Feed-Forward Network
        self.norm1 = nn.LayerNorm(d_model)  # Нормализация после attention
        self.norm2 = nn.LayerNorm(d_model)  # Нормализация после FFN
        self.dropout1 = nn.Dropout(dropout)  # Dropout для attention
        self.dropout2 = nn.Dropout(dropout)  # Dropout для FFN
    
    def forward(self, x, mask=None):
        # Pre-Norm: Norm -> Attention -> Add -> Norm -> FFN -> Add
        attn_out, attn_weights = self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask)  # Self-attention с Pre-Norm
        x = x + self.dropout1(attn_out)  # Остаточная связь + Dropout
        
        ffn_out = self.ffn(self.norm2(x))  # FFN с Pre-Norm
        x = x + self.dropout2(ffn_out)  # Остаточная связь после FFN
        
        return x, attn_weights  # Выход + веса внимания

block = TransformerEncoderBlock(d_model=32, n_heads=4)  # Создаём блок энкодера

In [5]:

batch, seq_len, d_model = 4, 16, 32  # Размерности для теста
x = torch.randn(batch, seq_len, d_model)  # Случайные эмбеддинги

output, attn_weights = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention:    {attn_weights.shape}")
print(f"Input is output shape match: {x.shape == output.shape}")
print(f"Output differs from input:   {not torch.allclose(x, output, atol=1e-4)}")

TypeError: masked_fill() received an invalid combination of arguments - got (bool, float), but expected one of:
 * (Tensor mask, Tensor value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)
 * (Tensor mask, Number value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)


## 7.5 Стекирование блоков: TransformerEncoder

Несколько энкодерных блоков, stacked последовательно.

In [6]:

class TransformerEncoder(nn.Module):  # Стопка энкодерных блоков
    """Стопка TransformerEncoderBlock."""
    
    def __init__(self, num_layers, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([  # Создаём num_layers блоков
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
        ])
    
    def forward(self, x, mask=None):  # Прямой проход через все слои
        all_attentions = []  # Собираем веса внимания со всех слоёв
        for i, layer in enumerate(self.layers):  # Проход по слоям
            x, attn = layer(x, mask)  # Прямой проход через блок
            all_attentions.append(attn)  # Сохраняем attention для визуализации
        return x, all_attentions  # Выход энкодера + веса внимания

encoder = TransformerEncoder(num_layers=6, d_model=64, n_heads=8)  # 6 слоёв, 8 голов
print(f"Encoder layers: {len(encoder.layers)}")
print(f"Total params: {sum(p.numel() for p in encoder.parameters()):,}")

Encoder layers: 1
Total params: 49,728


In [7]:

batch, seq_len, d_model = 2, 12, 64  # Размерности для теста
x = torch.randn(batch, seq_len, d_model)
output, attentions = encoder(x)

print(f"Encoder input:              {x.shape}")
print(f"Encoder output:             {output.shape}")
print(f"Number of attention maps:   {len(attentions)}")
print(f"Per-layer attention shape:  {attentions[0].shape}")

TypeError: masked_fill() received an invalid combination of arguments - got (bool, float), but expected one of:
 * (Tensor mask, Tensor value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)
 * (Tensor mask, Number value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)


## 7.6 Визуализация: активации после каждого компонента

Посмотрим, как меняются эмбеддинги после каждого блока.

In [8]:

encoder_small = TransformerEncoder(num_layers=4, d_model=16, n_heads=2, dropout=0.0)  # Маленький энкодер для визуализации
x = torch.randn(1, 8, 16)

layer_outputs = [x]  # Сохраняем вход как первый элемент
current = x
for i, layer in enumerate(encoder_small.layers):  # Проход по всем слоям
    current, attn = layer(current)  # Прямой проход через блок
    layer_outputs.append(current.detach())

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < len(layer_outputs):
        im = ax.imshow(layer_outputs[i][0].numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'After block {i}' if i > 0 else 'Input embeddings')
        ax.set_xlabel('d_model')
        ax.set_ylabel('Token position')
        plt.colorbar(im, ax=ax, fraction=0.046)
axes.flat[-1].axis('off')
plt.suptitle('Activations Flow Through Encoder Layers')
plt.tight_layout()
plt.show()

TypeError: masked_fill() received an invalid combination of arguments - got (bool, float), but expected one of:
 * (Tensor mask, Tensor value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)
 * (Tensor mask, Number value)
      didn't match because some of the arguments have invalid types: (!bool!, !float!)


In [16]:
print("=== Transformer Encoder complete ===")
print("Topics covered:")
print("  - Layer Normalization from scratch")
print("  - Feed-Forward Network: Linear -> GELU -> Linear")
print("  - Residual connections (Pre-Norm architecture)")
print("  - Multi-Head Self-Attention in encoder")
print("  - TransformerEncoderBlock with Add & Norm")
print("  - Stacked TransformerEncoder")
print("  - Activation flow visualization through layers")

=== Transformer Encoder complete ===
Topics covered:
  - Layer Normalization from scratch
  - Feed-Forward Network: Linear -> GELU -> Linear
  - Residual connections (Pre-Norm architecture)
  - Multi-Head Self-Attention in encoder
  - TransformerEncoderBlock with Add & Norm
  - Stacked TransformerEncoder
  - Activation flow visualization through layers
